# GW170817 PE — BlackJAX Nested Sampling, Fixed sky (NGC 4993), 128 s segments

Parameter estimation of GW170817 with **sky location fixed** to the known EM counterpart (NGC 4993):
$$\alpha = 3.44616\;\mathrm{rad}, \quad \delta = -0.408084\;\mathrm{rad}$$

- **Waveform**: `mlgw_bns_jax` (JAX-based BNS approximant)
- **Sampler**: [BlackJAX-NS](https://github.com/mrosep/blackjax_ns_gw) — GPU-accelerated nested sampling with a bilby-like acceptance-walk kernel
- **Data**: 1024 s of GWOSC strain for H1, L1, V1 — L1 deglitched via `gwpy.TimeSeries.gate()`
- **Segment duration**: 128 s (matching the paper, $\Delta f \approx 0.0078$ Hz)
- **Frequency range**: $[23, 2000]$ Hz

This notebook is the **BlackJAX-NS** counterpart of the SHARPy SMC fixed-sky notebook.
It reuses SHARPy's `GWNetwork` for data loading and PSD estimation but replaces the sampler with the GPU-accelerated acceptance-walk nested sampler from [Prathaban et al. (arXiv:2509.04336)](https://arxiv.org/abs/2509.04336).

**11 sampled parameters** (RA and Dec fixed):

| Index | Parameter | Prior range | Boundary |
|:---:|---|---|---|
| 0 | $\ln d_L$ | $[\ln 1, \ln 75]$ | reflective |
| 1 | $\theta_{JN}$ (inclination) | $[0, \pi]$ | reflective |
| 2 | $\phi_c$ (phase) | $[0, 2\pi]$ | periodic |
| 3 | $\psi$ (polarisation) | $[0, \pi]$ | periodic |
| 4 | $\mathcal{M}_c$ (chirp mass) | $[1.18, 1.21]\,M_\odot$ | reflective |
| 5 | $q$ (mass ratio) | $[0.5, 1.0]$ | reflective |
| 6 | $t_c$ (coalescence time) | $[-0.1, 0.1]\,\mathrm{s}$ | reflective |
| 7 | $\chi_1$ (spin 1) | $[-0.5, 0.5]$ | reflective |
| 8 | $\chi_2$ (spin 2) | $[-0.5, 0.5]$ | reflective |
| 9 | $\Lambda_1$ (tidal 1) | $[5, 5000]$ | reflective |
| 10 | $\Lambda_2$ (tidal 2) | $[5, 5000]$ | reflective |

## Environment setup (Colab / fresh environment)

This cell installs all required packages and clones the repositories. **Skip if running locally** with everything already installed.

**Note**: BlackJAX-NS requires the `nested_sampling` branch of BlackJAX from `handley-lab/blackjax` (pinned to a specific commit for API compatibility).

In [ ]:
import os, subprocess, sys

COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/mlgw_bns_jax" if COLAB else os.getcwd()

if COLAB:
    # ── Install JAX with CUDA 12 support ─────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "jax[cuda12]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    # ── Install other Python packages ────────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "corner", "gwpy", "h5py", "tqdm", "anesthetic",
    ])
    # ── Install BlackJAX nested_sampling branch (pinned commit) ──────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "blackjax@git+https://github.com/handley-lab/blackjax.git@dedbf11da33eb5ca286f6731e2c51f2b254b953f",
    ])

    # ── Clone the main repo (contains model, waveform loader, etc.) ──
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", "blackjax_ns_gw_pe", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git",
            REPO_DIR,
        ])

    # ── Clone blackjax_ns_gw (custom kernels) ────────────────────────
    blackjax_ns_gw_repo = os.path.join(REPO_DIR, "_blackjax_ns_gw_repo")
    kernels_src = os.path.join(blackjax_ns_gw_repo, "src", "custom_kernels")
    kernels_link = os.path.join(REPO_DIR, "custom_kernels")
    if not os.path.isdir(blackjax_ns_gw_repo):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/mrosep/blackjax_ns_gw.git",
            blackjax_ns_gw_repo,
        ])
    # Symlink so `from custom_kernels import ...` works
    if not os.path.exists(kernels_link):
        os.symlink(kernels_src, kernels_link)

    # ── Clone SHARPy (for data loading / GWNetwork only) ─────────────
    sharpy_repo = os.path.join(REPO_DIR, "_sharpy_repo")
    sharpy_pkg  = os.path.join(sharpy_repo, "sharpy")
    sharpy_link = os.path.join(REPO_DIR, "sharpy")
    if not os.path.isdir(sharpy_repo):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/sharpy.git",
            sharpy_repo,
        ])
    if not os.path.exists(sharpy_link):
        os.symlink(sharpy_pkg, sharpy_link)

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
    print(f"BlackJAX-NS kernels: {kernels_link} -> {kernels_src}")
    print(f"SHARPy (data loader): {sharpy_link} -> {sharpy_pkg}")
else:
    print("Not running on Colab — skipping setup.")

## Download & deglitch GWOSC data

Downloads 1024 s of 4 kHz strain from GWOSC for H1, L1 and V1. The L1 scatter-light glitch near the merger is removed using `gwpy`'s auto-gating. **Skip if the cleaned files already exist.**

In [ ]:
import os, time
import numpy as np

_GPS_START = 1187008114
_DURATION  = 1024
_SRATE     = 4096
_DATA_DIR  = "gw170817_data"
os.makedirs(_DATA_DIR, exist_ok=True)

_DETECTORS = ["H1", "L1", "V1"]

# Check if cleaned files already exist
_all_exist = all(
    os.path.isfile(os.path.join(_DATA_DIR,
        f"{d[0]}-{d}_CLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt"))
    for d in _DETECTORS
)

if _all_exist:
    print("Cleaned data files already exist — skipping download.")
else:
    from gwpy.timeseries import TimeSeries

    for det in _DETECTORS:
        raw_file = os.path.join(_DATA_DIR,
            f"{det[0]}-{det}_GWOSC_4KHZ_R1-{_GPS_START}-{_DURATION}.txt")
        out_file = os.path.join(_DATA_DIR,
            f"{det[0]}-{det}_CLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt")

        # Download raw data if needed
        if not os.path.isfile(raw_file):
            print(f"{det}: downloading {_DURATION}s from GWOSC...", flush=True)
            t0 = time.time()
            ts = TimeSeries.fetch_open_data(
                det, _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)
            strain = ts.value
            with open(raw_file, 'w') as f:
                f.write(f"# Gravitational wave strain for GW170817 for {det} (from GWOSC)\n")
                f.write(f"# This file has {_SRATE} samples per second\n")
                f.write(f"# starting GPS {_GPS_START} duration {_DURATION}\n")
                for val in strain:
                    f.write(f"{val:.16e}\n")
            print(f"  -> saved raw in {time.time()-t0:.1f}s")
        else:
            strain = np.loadtxt(raw_file, comments="#")
            ts = TimeSeries(strain, sample_rate=_SRATE, t0=_GPS_START)

        # Deglitch L1, pass through H1/V1
        if det == "L1":
            print(f"{det}: auto-gating L1 glitch...", flush=True)
            ts_clean = ts.gate(tzero=0.5, tpad=0.25, whiten=True, threshold=50.0)
        else:
            ts_clean = ts

        # Save cleaned file
        with open(out_file, 'w') as f:
            f.write(f"# Cleaned strain for GW170817 for {det} (GWOSC + glitch gate)\n")
            f.write(f"# {_SRATE} samples per second\n")
            f.write(f"# starting GPS {_GPS_START} duration {_DURATION}\n")
            for val in ts_clean.value:
                f.write(f"{val:.16e}\n")
        print(f"{det}: saved {os.path.basename(out_file)}")

    print("All detectors ready.")

In [ ]:
from __future__ import annotations

import os, sys, time
from functools import partial

import numpy as np

# Use GPU if available on Colab, otherwise CPU
if "google.colab" not in sys.modules:
    os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import jax.numpy as jnp
from jax import flatten_util  # needed for ravel_pytree
jax.config.update("jax_enable_x64", True)

print("JAX devices:", jax.devices())

## Load the waveform model and monkey-patch SHARPy

We replace the original `IMRPhenomD` template in SHARPy with our `mlgw_bns_jax` BNS waveform model **without modifying any SHARPy source file**. SHARPy is used only for data loading and the `GWNetwork` infrastructure.

In [ ]:
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = load_predict(MODEL_PATH)

# ---- Monkey-patch SHARPy's template -----------------------------------
import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses


def _template_mlgw_bns(params, frequency_array):
    """mlgw_bns_jax waveform, drop-in replacement for SHARPy's template."""
    mc, q = params[6], params[7]
    m1, m2 = McQ2Masses(mc, q)
    total_mass = m1 + m2
    chi1, chi2 = params[9], params[10]
    lambda_1, lambda_2 = params[11], params[12]
    phic = params[4]
    dist_mpc = jnp.exp(params[2])
    inclination = params[3]

    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass,
        distance_mpc=dist_mpc,
        inclination=inclination,
    )
    phase_factor = jnp.exp(-1j * phic)
    return hp * phase_factor, hc * phase_factor


_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
import sharpy.PSDs

print("Model loaded — SHARPy template patched with mlgw_bns_jax.")

## Event parameters

In [ ]:
TRIGGER_TIME = 1187008882.43
SEGMENT_DURATION = 128.0        # paper-matching segment length
SAMPLING_RATE = 4096
F_LOWER = 23.0
F_UPPER = 2000.0
DATA_START_GPS = 1187008114     # 1024s data file start
DATA_DURATION = 1024            # total data length (s)

# Fixed sky location: NGC 4993 (EM counterpart of GW170817)
FIXED_RA  = 3.44616     # rad
FIXED_DEC = -0.408084   # rad

DATA_DIR = "gw170817_data"
OUTDIR = "outdir_GW170817_blackjax_ns_fixedsky"
LABEL = "GW170817_blackjax_ns_fixedsky"
os.makedirs(OUTDIR, exist_ok=True)

print(f"Segment duration: {SEGMENT_DURATION}s  →  Δf = {1/SEGMENT_DURATION:.4f} Hz")
print(f"Data: {DATA_DURATION}s starting GPS {DATA_START_GPS}")
print(f"Fixed sky: RA = {FIXED_RA:.5f} rad, Dec = {FIXED_DEC:.6f} rad  (NGC 4993)")

## Load cleaned data and build detector network

We use 1024 s of GWOSC strain data downloaded via `gwpy.TimeSeries.fetch_open_data`, with the L1 scatter-light glitch removed by `gwpy`'s auto-gating.

With `SEGMENT_DURATION = 128 s`, SHARPy analyses a 128-s chunk centred on the trigger and uses the remaining ~896 s for Welch PSD estimation (~7 independent segments).

In [ ]:
# Cleaned 1024s files (L1 deglitched, H1/V1 as-is)
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_CLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_CLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_CLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing: {f}"
    print(f"{det}: {os.path.basename(f)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=data_files[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE,
        f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print(f"\nBuilding GW network (segment={SEGMENT_DURATION}s)...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

## Q-transform spectrograms

Verify the data quality: compare raw vs deglitched L1, and show all three cleaned detectors around the merger time.

In [ ]:
from gwpy.timeseries import TimeSeries
import matplotlib.pyplot as plt

MERGER_GPS = TRIGGER_TIME
WINDOW = 6.0
T_START_PLOT = MERGER_GPS - WINDOW / 2
T_END_PLOT   = MERGER_GPS + WINDOW / 2
F_MIN, F_MAX = 20.0, 800.0
Q_RANGE = (4, 64)

DET_COLORS = {"H1": "Reds", "L1": "Blues", "V1": "Purples"}
DET_LABELS = {"H1": "LIGO Hanford (H1)", "L1": "LIGO Livingston (L1)", "V1": "Virgo (V1)"}

def _qtransform(filepath):
    strain = np.loadtxt(filepath, comments="#")
    ts = TimeSeries(strain, sample_rate=SAMPLING_RATE, t0=DATA_START_GPS)
    ts_w = ts.whiten(4, 2)
    ts_c = ts_w.crop(T_START_PLOT - 1, T_END_PLOT + 1)
    return ts_c.q_transform(frange=(F_MIN, F_MAX), qrange=Q_RANGE,
                            outseg=(T_START_PLOT, T_END_PLOT), logf=True)

# ── L1 raw vs cleaned comparison ─────────────────────────────────────
raw_file = os.path.join(DATA_DIR,
    f"L-L1_GWOSC_4KHZ_R1-{DATA_START_GPS}-{DATA_DURATION}.txt")

if os.path.isfile(raw_file):
    qt_raw = _qtransform(raw_file)
    qt_cln = _qtransform(data_files["L1"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6), sharey=True)
    for ax, qt, title in [(ax1, qt_raw, "L1 — Raw (with glitch)"),
                           (ax2, qt_cln, "L1 — Cleaned (gated)")]:
        pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                            qt.value.T, cmap="Blues", vmin=0, vmax=25)
        ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
        ax.set_xlabel("Time relative to merger [s]", fontsize=13)
        ax.set_title(title, fontsize=14)
        ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7, label="Merger")
        ax.legend(loc="upper left"); ax.tick_params(labelsize=11)
        fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
    ax1.set_ylabel("Frequency [Hz]", fontsize=13)
    fig.suptitle("GW170817 — L1 glitch comparison (1024 s data)", fontsize=15)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{LABEL}_L1_comparison.png"), dpi=150)
    plt.show()
else:
    print(f"Raw L1 file not found ({raw_file}) — skipping glitch comparison.")

# ── All detectors cleaned ────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)
for ax, det in zip(axes, ["H1", "L1", "V1"]):
    qt = _qtransform(data_files[det])
    pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                        qt.value.T, cmap=DET_COLORS[det], vmin=0, vmax=25)
    ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
    ax.set_ylabel("Frequency [Hz]", fontsize=13)
    ax.set_title(f"{DET_LABELS[det]} (cleaned)", fontsize=13)
    ax.tick_params(labelsize=11)
    ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7)
    fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
axes[-1].set_xlabel("Time relative to merger [s]", fontsize=13)
fig.suptitle("GW170817 — Q-transform spectrograms (cleaned 1024 s data)",
             fontsize=15, y=0.995)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_qtransform_all.png"), dpi=150)
plt.show()

print("Spectrograms saved.")

## Define likelihood function

Build the log-likelihood using SHARPy's batched detector infrastructure, with RA/Dec fixed to NGC 4993.

$$\ln\mathcal{L} = -2\frac{\Delta t}{N}\sum_{\text{det}} \sum_{k} \frac{|d_k - h_k(\theta)|^2}{S_k}$$

The BlackJAX-NS sampler expects a **physical-space** log-likelihood function that takes a pytree (dict) of parameters.

In [ ]:
batched_detector = gw_network.batched_detector
log_likelihood_full = partial(log_likelihood_det, detector_list=batched_detector)

# ── Parameter names (11 sampled, matching the SHARPy fixed-sky notebook) ──
parameter_names = [
    "logdistance", "theta_jn", "phiref", "pol",
    "mc", "q", "tc", "chi1", "chi2", "lambda_1", "lambda_2",
]


def loglikelihood_from_dict(params_dict):
    """Log-likelihood wrapper: dict of 11 params → SHARPy 13-param array.

    Inserts fixed RA/Dec and calls the full SHARPy likelihood.
    """
    params_13 = jnp.array([
        FIXED_RA,                   # [0]  ra  (fixed)
        FIXED_DEC,                  # [1]  dec (fixed)
        params_dict["logdistance"], # [2]
        params_dict["theta_jn"],    # [3]
        params_dict["phiref"],      # [4]
        params_dict["pol"],         # [5]
        params_dict["mc"],          # [6]
        params_dict["q"],           # [7]
        params_dict["tc"],          # [8]
        params_dict["chi1"],        # [9]
        params_dict["chi2"],        # [10]
        params_dict["lambda_1"],    # [11]
        params_dict["lambda_2"],    # [12]
    ])
    return log_likelihood_full(params_13)


print(f"Fixed: RA = {FIXED_RA:.5f}, Dec = {FIXED_DEC:.6f}")
print(f"Sampling {len(parameter_names)} parameters: {parameter_names}")

## Define prior distributions and unit-cube transforms

BlackJAX-NS operates in the **unit hypercube** $[0,1]^n$. We define:
1. `prior_transform_fn` — maps unit cube to physical parameter space
2. `logprior_fn` — evaluates log-prior density in physical space

All priors here are **uniform**, matching the SHARPy fixed-sky notebook:
- Mass prior **flat in component masses** $m_{1,2}$, sampled in $(\mathcal{M}_c, q)$
- Aligned spins $|\chi_{1,2}| \leq 0.5$
- Tidal deformabilities $\Lambda_{1,2} \in [5, 5000]$
- Luminosity distance $D_L \in [1, 75]$ Mpc (flat in $\ln D_L$)

In [ ]:
# ── Prior bounds (same as the SHARPy fixed-sky notebook) ──────────────
param_bounds = {
    "logdistance": (jnp.log(1.0),  jnp.log(75.0)),   # ln(D_L / Mpc)
    "theta_jn":    (0.0,            jnp.pi),           # inclination
    "phiref":      (0.0,            2 * jnp.pi),       # phase (periodic)
    "pol":         (0.0,            jnp.pi),            # polarisation (periodic)
    "mc":          (1.18,           1.21),              # chirp mass [M_sun]
    "q":           (0.5,            1.0),               # mass ratio
    "tc":          (-0.1,           0.1),               # coalescence time [s]
    "chi1":        (-0.5,           0.5),               # spin 1
    "chi2":        (-0.5,           0.5),               # spin 2
    "lambda_1":    (5.0,            5000.0),            # tidal 1
    "lambda_2":    (5.0,            5000.0),            # tidal 2
}

# Periodic parameters (wraparound in unit cube)
periodic_params = {"phiref", "pol"}

# Pre-compute arrays in the order of parameter_names (for vectorised transforms)
param_mins = jnp.array([param_bounds[k][0] for k in parameter_names])
param_maxs = jnp.array([param_bounds[k][1] for k in parameter_names])

n_dims = len(parameter_names)


# ── Prior transform: unit cube → physical space ──────────────────────
@jax.jit
def prior_transform_fn(u_params):
    """Map unit-cube dict {name: scalar} → physical-space dict."""
    u_values, _ = jax.flatten_util.ravel_pytree(u_params)
    # All priors are uniform: x = min + u * (max - min)
    x_values = param_mins + u_values * (param_maxs - param_mins)
    example = {key: 0.0 for key in parameter_names}
    _, unflatten_fn = jax.flatten_util.ravel_pytree(example)
    return unflatten_fn(x_values)


# ── Log-prior in physical space ──────────────────────────────────────
@jax.jit
def logprior_fn(params):
    """Uniform log-prior: 0 inside bounds, -inf outside."""
    param_values, _ = jax.flatten_util.ravel_pytree(params)
    in_bounds = jnp.all((param_values >= param_mins) & (param_values <= param_maxs))
    log_vol = jnp.sum(jnp.log(param_maxs - param_mins))
    return jnp.where(in_bounds, -log_vol, -jnp.inf)


# ── Verify ravel order matches parameter_names ───────────────────────
# JAX sorts dict keys alphabetically; we need the mapping to be correct.
_example = {key: float(i) for i, key in enumerate(parameter_names)}
_flat, _ = jax.flatten_util.ravel_pytree(_example)
_ravel_order = []
for val in _flat:
    for key, test_val in _example.items():
        if abs(val - test_val) < 1e-10:
            _ravel_order.append(key)
            break
print(f"JAX ravel order: {_ravel_order}")

# Re-order param_mins/param_maxs to match JAX's ravel order
param_mins = jnp.array([param_bounds[k][0] for k in _ravel_order])
param_maxs = jnp.array([param_bounds[k][1] for k in _ravel_order])

print(f"Bounds check — log(D_L): [{param_bounds['logdistance'][0]:.3f}, {param_bounds['logdistance'][1]:.3f}]")
print(f"n_dims = {n_dims}")

## Configure BlackJAX-NS nested sampler

Set up the GPU-accelerated acceptance-walk nested sampler from [blackjax_ns_gw](https://github.com/mrosep/blackjax_ns_gw).

Key configuration:
- **`n_live`**: Number of live points (governs posterior resolution and evidence accuracy)
- **`n_delete`**: Batch size — how many dead points are removed per iteration. Set to `n_live // 2` for GPU parallelisation
- **`n_target`**: Target accepted MCMC steps before replacing a dead point (Bilby default: 60)
- **`max_mcmc`**: Hard budget on MCMC steps per replacement
- **Termination**: Stop when the estimated remaining log-evidence $\Delta\ln\mathcal{Z} < 0.1$

In [ ]:
import tqdm

# Import BlackJAX-NS custom kernels
from custom_kernels import (
    acceptance_walk_sampler,
    create_unit_cube_functions,
    init_unit_cube_particles,
    transform_to_physical,
)

# ── Sampler hyper-parameters ──────────────────────────────────────────
N_LIVE = 1400
N_DELETE = N_LIVE // 2          # batch size for GPU parallelisation
N_TARGET = 60                   # target accepted walks per chain
MAX_MCMC = 5000                 # hard budget per replacement
SEED = 42
DLOGZ_STOP = 0.1               # termination threshold

rng_key = jax.random.PRNGKey(SEED)
rng_key, init_key = jax.random.split(rng_key, 2)

# ── Example parameter structure ───────────────────────────────────────
example_params = {key: 0.0 for key in parameter_names}

# ── Initialize live particles in unit hypercube [0,1]^n ───────────────
unit_cube_particles = init_unit_cube_particles(init_key, example_params, N_LIVE)

# ── Periodic mask for wraparound parameters ───────────────────────────
periodic_mask = jax.tree_util.tree_map(lambda _: False, example_params)
for key in parameter_names:
    if key in periodic_params:
        periodic_mask[key] = True

# ── Create unit-cube wrapper functions ────────────────────────────────
unit_cube_fns = create_unit_cube_functions(
    physical_loglikelihood_fn=loglikelihood_from_dict,
    prior_transform_fn=prior_transform_fn,
    mask_tree=periodic_mask,
)

# ── Build the nested sampler ──────────────────────────────────────────
nested_sampler = acceptance_walk_sampler(
    logprior_fn=unit_cube_fns["logprior_fn"],
    loglikelihood_fn=unit_cube_fns["loglikelihood_fn"],
    nlive=N_LIVE,
    n_target=N_TARGET,
    max_mcmc=MAX_MCMC,
    num_delete=N_DELETE,
    stepper_fn=unit_cube_fns["stepper_fn"],
)

# ── Initialize sampler state ──────────────────────────────────────────
state = nested_sampler.init(unit_cube_particles)

print(f"BlackJAX-NS configured: {N_LIVE} live points, batch size {N_DELETE}")
print(f"Target walks: {N_TARGET}, max MCMC: {MAX_MCMC}")
print(f"Termination: ΔlnZ < {DLOGZ_STOP}")

## Run nested sampling

Execute the nested sampling loop using JAX's JIT compilation for performance. Each iteration removes `N_DELETE` dead points and replaces them via the acceptance-walk kernel. The loop terminates when $\Delta\ln\mathcal{Z} < 0.1$.

The log-evidence $\ln\mathcal{Z}$ accumulates from the dead-point likelihoods and the shrinking prior volume at each iteration:
$$\ln\mathcal{Z} = \ln\sum_i w_i \mathcal{L}_i$$

In [ ]:
@jax.jit
def one_step(carry, xs):
    """Single nested sampling iteration (JIT-compiled for speed)."""
    state, k = carry
    k, subk = jax.random.split(k, 2)
    state, dead_point = nested_sampler.step(subk, state)
    return (state, k), dead_point


def terminate(state):
    """Termination condition: stop when remaining evidence is small."""
    dlogz = jnp.logaddexp(0, state.logZ_live - state.logZ)
    return jnp.isfinite(dlogz) and dlogz < DLOGZ_STOP


print(f"Starting BlackJAX-NS nested sampling with {N_LIVE} live points...")
print(f"Batch size: {N_DELETE} dead points per iteration")
print(f"Termination: ΔlnZ < {DLOGZ_STOP}")

start = time.time()
dead = []

with tqdm.tqdm(desc="Dead points", unit=" pts") as pbar:
    while not terminate(state):
        (state, rng_key), dead_info = one_step((state, rng_key), None)
        dead.append(dead_info)
        pbar.update(N_DELETE)

dt = time.time() - start
n_dead_total = len(dead) * N_DELETE
print(f"\nDone in {dt:.1f} s — {n_dead_total} dead points generated")
print(f"Log evidence (running):  ln Z = {state.logZ:.2f}")
print(f"Log evidence (live):     ln Z_live = {state.logZ_live:.2f}")

## Extract posterior samples and evidence

Finalise the nested sampling run: combine dead points with the remaining live points, transform from the unit hypercube back to physical parameter space, and compute the Bayesian evidence $\ln\mathcal{Z}$.

In [ ]:
import pickle
from blackjax.ns.utils import finalise
from anesthetic import NestedSamples

# ── Finalise: merge dead points + remaining live points ───────────────
final_state = finalise(state, dead)

# ── Save final state ──────────────────────────────────────────────────
state_path = os.path.join(OUTDIR, f"{LABEL}_final_state.pkl")
with open(state_path, "wb") as f:
    pickle.dump(final_state, f)
print(f"Saved final state to {state_path}")

# ── Transform samples from unit cube → physical space ─────────────────
physical_particles = transform_to_physical(final_state.particles, prior_transform_fn)

# ── Build anesthetic NestedSamples object ─────────────────────────────
column_to_label = {
    "logdistance": r"$\ln d_L$",
    "theta_jn":    r"$\theta_{JN}$",
    "phiref":      r"$\phi_c$",
    "pol":         r"$\psi$",
    "mc":          r"$\mathcal{M}_c$",
    "q":           r"$q$",
    "tc":          r"$t_c$",
    "chi1":        r"$\chi_1$",
    "chi2":        r"$\chi_2$",
    "lambda_1":    r"$\Lambda_1$",
    "lambda_2":    r"$\Lambda_2$",
}

# Handle potential NaN values in log-likelihood birth
logL_birth = final_state.loglikelihood_birth.copy()
logL_birth = jnp.where(jnp.isnan(logL_birth), -jnp.inf, logL_birth)

samples_anesthetic = NestedSamples(
    physical_particles,
    logL=final_state.loglikelihood,
    logL_birth=logL_birth,
    labels=column_to_label,
    logzero=jnp.nan,
    dtype=jnp.float64,
)

# Save results to CSV
csv_path = os.path.join(OUTDIR, f"{LABEL}_results.csv")
samples_anesthetic.to_csv(csv_path)
print(f"Saved posterior samples to {csv_path}")

# ── Evidence summary ──────────────────────────────────────────────────
logZ_samples = samples_anesthetic.logZ(100)
print(f"\nBayesian evidence:")
print(f"  ln Z (from integrator): {state.logZ:.2f}")
print(f"  ln Z (anesthetic):      {logZ_samples.mean():.2f} ± {logZ_samples.std():.2f}")
print(f"  Total dead points:      {len(dead) * N_DELETE}")

## Convert to equally-weighted samples for corner plots

Extract equally-weighted posterior samples from the `anesthetic` NestedSamples object for use with standard plotting libraries.

The physical particle dict is keyed by parameter name; we need to extract an ordered NumPy array matching `parameter_names`.

In [ ]:
# ── Build ordered sample array from physical_particles dict ───────────
# physical_particles is a dict of JAX arrays, each (n_total,)
# We need an (n_total, n_dims) array in parameter_names order.
n_total = len(final_state.loglikelihood)

samples_array = np.column_stack([
    np.array(physical_particles[key]) for key in parameter_names
])

# ── Compute posterior weights from nested sampling ────────────────────
logL = np.array(final_state.loglikelihood)
logL_birth_np = np.array(logL_birth)

# Posterior weights: w_i ∝ L_i × ΔX_i
# Approximate ΔX_i from nested sampling shrinkage
# For equally-weighted resampling, use anesthetic's built-in method
eq_samples = samples_anesthetic.sample(nsamples=5000)

# Map anesthetic columns back to parameter_names order
# anesthetic uses the ravel order from the dict
eq_array = np.column_stack([
    np.array(eq_samples[key]) for key in _ravel_order
])

# Reorder to parameter_names order
ravel_to_param_idx = [_ravel_order.index(k) for k in parameter_names]
samples = eq_array[:, ravel_to_param_idx]

print(f"Equally-weighted posterior samples: {samples.shape}")
print(f"Columns: {parameter_names}")

## Corner plot (all 11 parameters)

In [ ]:
from corner import corner

fig = corner(
    samples, show_titles=True,
    labels=parameter_names, title_kwargs={"fontsize": 10},
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
fig.savefig(plot_path, dpi=150)
print(f"Saved to {plot_path}")
fig

## Paper-style corner plot (Figure 9)

Compute derived parameters from the posterior samples and produce a corner plot matching Figure 9 of [arXiv:2210.15684](https://arxiv.org/abs/2210.15684), showing only:
- $\mathcal{M}_c$ — chirp mass
- $q$ — mass ratio
- $\chi_\text{eff}$ — effective spin parameter
- $\tilde{\Lambda}$ — reduced tidal deformability
- $D_L$ — luminosity distance [Mpc]

Column indices for the 11-parameter samples:
`[0] logdist, [1] incl, [2] phic, [3] pol, [4] mc, [5] q, [6] tc, [7] chi1, [8] chi2, [9] lambda_1, [10] lambda_2`

In [ ]:
from sharpy.utils import McQ2Masses

# Extract raw sampled parameters (11-param layout, parameter_names order)
mc_samples   = samples[:, 4]   # chirp mass
q_samples    = samples[:, 5]   # mass ratio (m2/m1 <= 1)
chi1_samples = samples[:, 7]   # spin 1
chi2_samples = samples[:, 8]   # spin 2
lam1_samples = samples[:, 9]   # Lambda_1
lam2_samples = samples[:, 10]  # Lambda_2
logd_samples = samples[:, 0]   # log distance

# Compute component masses
m1_samples = np.zeros(len(mc_samples))
m2_samples = np.zeros(len(mc_samples))
for i in range(len(mc_samples)):
    m1_samples[i], m2_samples[i] = McQ2Masses(mc_samples[i], q_samples[i])

# chi_eff = (m1*chi1 + m2*chi2) / (m1 + m2)
chi_eff_samples = (m1_samples * chi1_samples + m2_samples * chi2_samples) / (m1_samples + m2_samples)

# Lambda_tilde (reduced tidal deformability)
M_samples = m1_samples + m2_samples
lambda_tilde_samples = (16.0 / 13.0) * (
    (m1_samples + 12.0 * m2_samples) * m1_samples**4 * lam1_samples
    + (m2_samples + 12.0 * m1_samples) * m2_samples**4 * lam2_samples
) / M_samples**5

# D_L in Mpc
dL_samples = np.exp(logd_samples)

# Build the 5-parameter array for the corner plot
paper_samples = np.column_stack([
    mc_samples,
    q_samples,
    chi_eff_samples,
    lambda_tilde_samples,
    dL_samples,
])

paper_labels = [
    r"$\mathcal{M}_c$ $[M_\odot]$",
    r"$q$",
    r"$\chi_{\rm eff}$",
    r"$\tilde{\Lambda}$",
    r"$D_L$ [Mpc]",
]

fig_paper = corner(
    paper_samples, show_titles=True,
    labels=paper_labels,
    title_kwargs={"fontsize": 12},
    quantiles=[0.05, 0.5, 0.95],
    levels=(0.5, 0.9),
    fill_contours=True,
    color="tab:blue",
)
plot_path_paper = os.path.join(OUTDIR, f"{LABEL}_corner_paper.png")
fig_paper.savefig(plot_path_paper, dpi=150)
print(f"Saved to {plot_path_paper}")
fig_paper

## Waveform recovery

Draw random posterior samples and generate corresponding waveforms. Plot these against the data in the frequency domain to visually assess the quality of the parameter estimation.

In [ ]:
# ── Draw a few posterior waveforms and compare with the data ──────────
N_DRAW = 50
rng_draw = np.random.default_rng(123)
draw_idx = rng_draw.choice(len(samples), size=N_DRAW, replace=False)

# Frequency array from the detector network
freq_array = np.array(batched_detector.Frequency[0])  # same for all detectors

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
det_names = ["H1", "L1", "V1"]

for i_det, (ax, det_name) in enumerate(zip(axes, det_names)):
    # Plot data
    data_fd = np.array(batched_detector.FrequencySeries[i_det])
    psd_fd  = np.array(batched_detector.PowerSpectralDensity[i_det])
    ax.semilogy(freq_array, np.abs(data_fd), color="gray", alpha=0.3, lw=0.5, label="Data")

    # Plot posterior waveforms
    for j, idx in enumerate(draw_idx):
        s = samples[idx]
        # Build 13-param array for the template
        params_13 = jnp.array([
            FIXED_RA, FIXED_DEC,
            s[0], s[1], s[2], s[3],   # logdist, theta_jn, phiref, pol
            s[4], s[5], s[6],          # mc, q, tc
            s[7], s[8],                # chi1, chi2
            s[9], s[10],               # lambda_1, lambda_2
        ])
        hp, hc = _template_mlgw_bns(params_13, jnp.array(freq_array))
        ax.semilogy(freq_array, np.abs(np.array(hp)), color="tab:blue",
                    alpha=0.08, lw=0.5, label="Posterior" if j == 0 else None)

    ax.set_ylabel(f"|h(f)| [{det_name}]", fontsize=12)
    ax.set_ylim(1e-25, 1e-21)
    ax.legend(loc="upper right", fontsize=10)
    ax.tick_params(labelsize=10)

axes[-1].set_xlabel("Frequency [Hz]", fontsize=12)
axes[-1].set_xlim(F_LOWER, 500)
fig.suptitle("GW170817 — Waveform recovery (BlackJAX-NS)", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_waveform_recovery.png"), dpi=150)
plt.show()
print("Waveform recovery plot saved.")